In [1]:
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from dotenv import load_dotenv
load_dotenv()

C:\Users\ayush\AppData\Local\Temp\ipykernel_19924\551219779.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


True

In [2]:
# ──────────────────────────────────────────────────────────────────
# SETUP: Create our sample company data
# ──────────────────────────────────────────────────────────────────

chunks = [
    "Microsoft acquired GitHub for 7.5 billion dollars in 2018.",
    "Tesla Cybertruck production ramp begins in 2024.",
    "Google is a large technology company with global operations.",
    "Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.",
    "SpaceX develops Starship rockets for Mars missions.",
    "The tech giant acquired the code repository platform for software development.",
    "NVIDIA designs Starship architecture for their new GPUs.",
    "Tesla Tesla Tesla financial quarterly results improved significantly.",
    "Cybertruck reservations exceeded company expectations.",
    "Microsoft is a large technology company with global operations.", 
    "Apple announced new iPhone features for developers.",
    "The apple orchard harvest was excellent this year.",
    "Python programming language is widely used in AI.",
    "The python snake can grow up to 20 feet long.",
    "Java coffee beans are imported from Indonesia.", 
    "Java programming requires understanding of object-oriented concepts.",
    "Orange juice sales increased during winter months.",
]

In [3]:
# Convert to document objects from langchain
documents = [
    Document(
        page_content=chunk,
        metadata={"source": f"chunk_{i}"}
    ) for i, chunk in enumerate(chunks)
]

print("Sample data:")
for i, chunk in enumerate(chunks, 1):
    print(f"{i}. {chunk}")
    
print("\n"+ "="*80)

Sample data:
1. Microsoft acquired GitHub for 7.5 billion dollars in 2018.
2. Tesla Cybertruck production ramp begins in 2024.
3. Google is a large technology company with global operations.
4. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
5. SpaceX develops Starship rockets for Mars missions.
6. The tech giant acquired the code repository platform for software development.
7. NVIDIA designs Starship architecture for their new GPUs.
8. Tesla Tesla Tesla financial quarterly results improved significantly.
9. Cybertruck reservations exceeded company expectations.
10. Microsoft is a large technology company with global operations.
11. Apple announced new iPhone features for developers.
12. The apple orchard harvest was excellent this year.
13. Python programming language is widely used in AI.
14. The python snake can grow up to 20 feet long.
15. Java coffee beans are imported from Indonesia.
16. Java pr

## Creating different types of retrievers
1. Vector Retriever
2. Keyword Retriever 
3. Hybrid Retriever

### Vector Embedding Retriever

In [5]:
print("Setting up vector retriever...")

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vectorStore = Chroma.from_documents(
    documents=documents,
    embedding=embedding_model,
    collection_metadata={
        "hnsw:space": "cosine"
    }
)

Setting up vector retriever...


In [6]:
vectorRetriever = vectorStore.as_retriever(search_kwargs={"k": 2})

# Test semantic search
test_query = "space exploration company"
print(f"Testing query: '{test_query}'")

test_docs = vectorRetriever.invoke(test_query)
for doc in test_docs:
    print(f"Found: {doc.page_content}")

Testing query: 'space exploration company'
Found: SpaceX develops Starship rockets for Mars missions.
Found: Cybertruck reservations exceeded company expectations.


### BM25 Retriever

In [16]:
print("Setting up BM25 retriever...")
bm25Retriever = BM25Retriever.from_documents(documents)
bm25Retriever.k = 3

Setting up BM25 retriever...


In [20]:
# Test exact keyword matching...

test_query = "Tesla"

print(f"Testing: '{test_query}'")

test_docs = bm25Retriever.invoke(test_query)
for doc in test_docs:
    print(f"Found: {doc.page_content}")

Testing: 'Tesla'
Found: Tesla Tesla Tesla financial quarterly results improved significantly.
Found: Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
Found: Tesla Cybertruck production ramp begins in 2024.


### Hybrid Retriever (Vector + Keyword Search)

In [ ]:
print("Setting up hybrid search...")

hybridRetriever = EnsembleRetriever(
    retrievers=[vectorRetriever, bm25Retriever],
    weights=[0.5, 0.5]
)

print("Setup complete\n")

Setting up hybrid search...
Setup complete



In [29]:
# Hybrid should combine both strengths for best results

test_query = "purchase cost 7.5 billion"

print(f"Testing: '{test_query}'")
retrievedChunks = hybridRetriever.invoke(test_query)
for i, doc in enumerate(retrievedChunks, 1):
    print(f"{i}. {doc.page_content}")
print()

test_query = "electric vehicle manufacturing Cybertruck"

print(f"Testing: '{test_query}'")
retrievedChunks = hybridRetriever.invoke(test_query)
for i, doc in enumerate(retrievedChunks, 1):
    print(f"{i}. {doc.page_content}")
print()

test_query = "company performance Tesla"

print(f"Testing: '{test_query}'")
retrievedChunks = hybridRetriever.invoke(test_query)
for i, doc in enumerate(retrievedChunks, 1):
    print(f"{i}. {doc.page_content}")
print()

Testing: 'purchase cost 7.5 billion'
1. Microsoft acquired GitHub for 7.5 billion dollars in 2018.
2. Cybertruck reservations exceeded company expectations.
3. Orange juice sales increased during winter months.
4. Java programming requires understanding of object-oriented concepts.

Testing: 'electric vehicle manufacturing Cybertruck'
1. Tesla Cybertruck production ramp begins in 2024.
2. Cybertruck reservations exceeded company expectations.
3. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.

Testing: 'company performance Tesla'
1. Tesla Tesla Tesla financial quarterly results improved significantly.
2. Tesla reported strong quarterly results. Tesla continues to lead in electric vehicles. Tesla announced new manufacturing facilities.
3. Cybertruck reservations exceeded company expectations.



In [28]:
# Combine the query and relevant document contents

from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

test_query = "company performance Tesla"
retrievedChunks = hybridRetriever.invoke(test_query)

combined_input = f"""
Based on the following documents, please answer this question: {test_query}

Documents:
{chr(10).join([f"- {doc.page_content}" for doc in retrievedChunks])}

Please provide a clear, helpful answer using only the information from these documents. If you can't find answer in these documents, say "I don't have enough information to answer that question based on the provided documents".
"""

model = ChatOpenAI(model="gpt-4o")

messages = [
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content=combined_input)
]

result = model.invoke(messages)

print("\n----Generated Response----")
print("Content Only:")
print(result.content)


----Generated Response----
Content Only:
Tesla's performance appears to be strong based on the provided documents. The company reported significantly improved financial quarterly results and strong quarterly results overall. Additionally, Tesla continues to lead in the electric vehicle market and has announced new manufacturing facilities, suggesting positive growth and expansion. The company has also exceeded expectations with reservations for the Cybertruck.
